# Cross-slice Tracking Evaluation with SAM + ZMemoryAdapter

This notebook mirrors the overall flow of `notebooks/inference_and_evaluation.ipynb`, but is specialized for **cross-slice cell tracking in 3D microscopy Z-stacks**.

We assume the following evaluation protocol:

1. Slice `z=0` is the initialization slice.
2. We use the **center point of the GT mask** on `z=0` as a positive point prompt.
3. For slices `z=1 ... z=Z-1`, we disable point and box prompts completely.
4. The model predicts each next slice **autoregressively** only from the previous slice memory state, through `ZMemoryAdapter`.
5. We evaluate the prediction quality with **per-slice IoU** and visualize the memory decay along the Z axis.


## Running this notebook

This notebook is meant to run inside the `micro-sam` repository.

Before running it on your server, please prepare:

- a 3D image volume in `.npy` format with shape `[Z, H, W]`
- a 3D GT mask volume in `.npy` format with shape `[Z, H, W]`
- your finetuned checkpoint such as `best.pt`
- optionally a separate memory adapter checkpoint such as `best_mem_adapter.pt`

Note:
The current implementation assumes that the object is already present on the first slice `z=0`. If your target appears later, you should crop or re-index the volume first.


## Import the libraries

In [ ]:
import json
import sys
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


## Load the cross-slice tracking utilities

We reuse the helper module from `notebooks/cross_slice_tracking_evaluation.py` so the notebook stays compact and easy to maintain.

In [ ]:
repo_root = Path.cwd()
if not (repo_root / 'notebooks').exists():
    repo_root = repo_root.parent

module_path = repo_root / 'notebooks' / 'cross_slice_tracking_evaluation.py'
assert module_path.exists(), f'Cannot find helper module: {module_path}'

spec = importlib.util.spec_from_file_location('cross_slice_tracking_module', module_path)
tracking_eval = importlib.util.module_from_spec(spec)
sys.modules['cross_slice_tracking_module'] = tracking_eval
spec.loader.exec_module(tracking_eval)

print(f'Loaded helper module from: {module_path}')


## Configure and load the volume data

Just like in the official notebook, we first define the paths in one place so the rest of the notebook can reuse them.

In [ ]:
data_root = repo_root / 'data' / 'cross_slice_tracking'  # overwrite this on your machine if needed
image_volume_path = data_root / 'image_volume.npy'
gt_volume_path = data_root / 'gt_volume.npy'

assert image_volume_path.exists(), f'Image volume not found: {image_volume_path}'
assert gt_volume_path.exists(), f'GT volume not found: {gt_volume_path}'

image_volume = np.load(image_volume_path)
gt_volume = np.load(gt_volume_path)

print(f'image_volume shape: {image_volume.shape}, dtype: {image_volume.dtype}')
print(f'gt_volume shape:    {gt_volume.shape}, dtype: {gt_volume.dtype}')


## Create convenience visualization functions

As in `inference_and_evaluation.ipynb`, it is useful to wrap repeated visualization logic into small helper functions.

In [ ]:
def plot_volume_samples(image_volume, gt_volume, z_indices=None):
    if z_indices is None:
        z_indices = np.linspace(0, image_volume.shape[0] - 1, num=min(4, image_volume.shape[0]), dtype=int)
    z_indices = list(dict.fromkeys(int(z) for z in z_indices))

    fig, axes = plt.subplots(len(z_indices), 2, figsize=(10, 3 * len(z_indices)))
    if len(z_indices) == 1:
        axes = np.asarray([axes])

    for row, z in enumerate(z_indices):
        axes[row, 0].imshow(image_volume[z], cmap='gray')
        axes[row, 0].set_title(f'Image z={z}')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(gt_volume[z], cmap='gray')
        axes[row, 1].set_title(f'GT z={z}')
        axes[row, 1].axis('off')

    plt.tight_layout()
    return fig, axes


def plot_tracking_predictions(image_volume, gt_volume, pred_volume, iou_per_slice, z_indices=None):
    if z_indices is None:
        z_indices = np.linspace(0, image_volume.shape[0] - 1, num=min(5, image_volume.shape[0]), dtype=int)
    z_indices = list(dict.fromkeys(int(z) for z in z_indices))

    fig, axes = plt.subplots(len(z_indices), 3, figsize=(12, 3 * len(z_indices)))
    if len(z_indices) == 1:
        axes = np.asarray([axes])

    for row, z in enumerate(z_indices):
        axes[row, 0].imshow(image_volume[z], cmap='gray')
        axes[row, 0].set_title(f'Image z={z}')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(gt_volume[z], cmap='gray')
        axes[row, 1].set_title(f'GT z={z}')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(image_volume[z], cmap='gray')
        axes[row, 2].imshow(pred_volume[z], cmap='jet', alpha=0.35)
        axes[row, 2].set_title(f'Prediction z={z} | IoU={iou_per_slice[z]:.3f}')
        axes[row, 2].axis('off')

    plt.tight_layout()
    return fig, axes


## Visualize the input samples

Before running the model, we inspect a few slices from the image volume and the corresponding GT masks.

In [ ]:
plot_volume_samples(image_volume, gt_volume)
plt.show()


## Configure the model and checkpoints

This cell follows the same spirit as the official notebook: first define the model type and checkpoint paths, then reuse the variables in the evaluation cells below.

In [ ]:
model_type = 'vit_b'  # overwrite with vit_l if needed
device = 'cuda'  # overwrite with 'cpu' if you do not have a GPU

base_sam_checkpoint_path = None
finetuned_checkpoint_path = Path('/path/to/best.pt')
adapter_checkpoint_path = Path('/path/to/best_mem_adapter.pt')  # set to None if not used

output_dir = repo_root / 'outputs' / 'cross_slice_tracking_eval'
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Output directory: {output_dir}')


## Initialize SAM and inject the memory adapter

This step loads the base SAM model through `micro-sam`, injects `ZMemoryAdapter` into `sam_model.memory_adapter`, and then loads the finetuned weights safely.

In [ ]:
sam_model, predictor, load_report = tracking_eval.load_sam_with_memory_adapter(
    model_type=model_type,
    device=device,
    base_sam_checkpoint_path=str(base_sam_checkpoint_path) if base_sam_checkpoint_path is not None else None,
    finetuned_checkpoint_path=str(finetuned_checkpoint_path) if finetuned_checkpoint_path is not None else None,
    adapter_checkpoint_path=str(adapter_checkpoint_path) if adapter_checkpoint_path is not None else None,
)
tracking_eval.print_load_report(load_report)


## Run the zero-shot cross-slice tracking evaluation

This is the core evaluation loop:

- `z=0`: use the GT center point as the initialization prompt
- `z>=1`: use only the previous memory state to generate dense prompts
- compute the IoU for every slice


In [ ]:
results = tracking_eval.evaluate_cross_slice_tracking(
    sam_model=sam_model,
    predictor=predictor,
    image_volume=image_volume,
    gt_volume=gt_volume,
    mask_threshold=0.5,
)

pred_volume = results['pred_volume']
iou_per_slice = results['iou_per_slice']
init_point = results['init_point']

print(f'Initialization point on z=0: {init_point.tolist()}')
print(f'Number of slices: {len(iou_per_slice)}')
print(f'Mean IoU: {np.mean(iou_per_slice):.4f}')
print(f'Min IoU:  {np.min(iou_per_slice):.4f}')
print(f'Max IoU:  {np.max(iou_per_slice):.4f}')


## Plot the memory decay curve

Like the evaluation plots in the official notebook, we summarize the performance visually. Here the X axis is the slice index and the Y axis is the IoU.

In [ ]:
curve_path = output_dir / 'iou_curve.png'
tracking_eval.plot_slice_iou_curve(
    iou_per_slice,
    title='Cross-slice Tracking IoU',
    save_path=str(curve_path),
)
plt.show()
print(f'Saved curve to: {curve_path}')


## Inspect the predictions on several slices

This qualitative inspection complements the IoU curve and makes it easier to see where the memory starts drifting.

In [ ]:
plot_tracking_predictions(
    image_volume=image_volume,
    gt_volume=gt_volume,
    pred_volume=pred_volume,
    iou_per_slice=iou_per_slice,
)
plt.show()


## Save the outputs

This final step stores the predicted mask volume, per-slice IoU values, and a small JSON summary for later analysis.

In [ ]:
pred_volume_path = output_dir / 'pred_volume.npy'
iou_path = output_dir / 'iou_per_slice.npy'
summary_path = output_dir / 'summary.json'

np.save(pred_volume_path, pred_volume.astype(np.uint8))
np.save(iou_path, np.asarray(iou_per_slice, dtype=np.float32))

summary = {
    'image_volume_path': str(image_volume_path),
    'gt_volume_path': str(gt_volume_path),
    'model_type': model_type,
    'device': device,
    'finetuned_checkpoint_path': str(finetuned_checkpoint_path) if finetuned_checkpoint_path is not None else None,
    'adapter_checkpoint_path': str(adapter_checkpoint_path) if adapter_checkpoint_path is not None else None,
    'mean_iou': float(np.mean(iou_per_slice)),
    'num_slices': int(len(iou_per_slice)),
    'init_point': init_point.tolist(),
    'load_report': load_report,
}
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Saved prediction volume to: {pred_volume_path}')
print(f'Saved IoU array to:        {iou_path}')
print(f'Saved summary to:          {summary_path}')


## What next?

If this notebook gives you a good memory decay profile, the next step is usually one of the following:

1. Run the same notebook over multiple tracked cells and compare the IoU curves.
2. Save the results for several checkpoints and compare their mean IoU and tail-slice IoU.
3. Add extra visualizations such as contour overlays or failure-case slice dumps.
